# 意图识别 — 基线方案 (baseline)

这是一个**最简提交**示例，用来给参赛选手提供一个明确的下限基线：

- 直接在 `datasets/train.jsonl`（16 行，每类 1 条）上微调 `bert-base-chinese`
- 最终生成 `submission.zip` (内含 `submission_val.jsonl` 与 `submission_test.jsonl`)

**注意**: 16 行训练数据 × 3 epochs 在 16 batch_size 下只有 ~3 个梯度步，BERT 远未收敛，分数会很低。本基线只为提交格式与流程做演示。

In [52]:
from __future__ import annotations

import json
import random
import zipfile
from pathlib import Path
from typing import Any

import numpy as np
import torch
import torch.nn as nn
from torch.optim.lr_scheduler import CosineAnnealingLR, LambdaLR
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
from tqdm import tqdm
import random

seed = 42

random.seed(seed)                  # Python built-in random
np.random.seed(seed)               # NumPy
torch.manual_seed(seed)            # PyTorch (CPU)
torch.cuda.manual_seed(seed)       # PyTorch (single GPU)
torch.cuda.manual_seed_all(seed)   # PyTorch (all GPUs)

# Ensures deterministic behavior
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False



DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"DEVICE = {DEVICE}")
PATH = '/bohr/train-z7v5/v1'
BASE_MODEL_NAME = "bert-base-chinese"
# MODEL_DIR = PATH + "models--bert-base-chinese/8f23c25b06e129b6c986331a13d8d025a92cf0ea"
MODEL_DIR = PATH + "/bert-base-chinese"
TRAIN_PATH = Path(PATH + "/train.jsonl")


MAX_LEN = 512
BATCH_SIZE = 16
EPOCHS = 120
LEARNING_RATE = 1e-5
SEED = 42


def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


set_seed(SEED)

In [53]:
INTENT_LABELS = [
    "游戏技巧", "游戏角色信息", "周边饭馆", "查找高端酒店", "查找地址",
    "健康知识", "查找医疗信息", "美容化妆技巧", "美食烹饪技巧", "软件开发问题",
    "软件使用问题", "查找小说剧情", "查找小说角色信息", "查找营业时间",
    "法律法条解释", "法律问题咨询",
]
LABEL_TO_ID = {l: i for i, l in enumerate(INTENT_LABELS)}
ID_TO_LABEL = {i: l for i, l in enumerate(INTENT_LABELS)}

In [54]:
def extract_user_utterance(text: str) -> str:
    user_utts = []
    for line in text.strip().split("\n"):
        if line.strip().startswith("usr:"):
            user_utts.append(line.strip()[4:].strip())
    return user_utts[-1] if user_utts else ""


def preprocess_input_text(text: Any) -> str:
    if text is None:
        return ""
    s = str(text).strip()
    if not s:
        return ""
    if "usr:" in s:
        u = extract_user_utterance(s)
        if u:
            return u
    return s


def load_jsonl(path):
    return [json.loads(l) for l in path.read_text(encoding="utf-8").splitlines() if l.strip()]


train_records = load_jsonl(TRAIN_PATH)
train_rows = []
for r in train_records:
    text = preprocess_input_text(r.get("input"))
    lab = str(r.get("label", "")).strip()
    if text and lab in LABEL_TO_ID:
        train_rows.append({"text": text, "label_id": LABEL_TO_ID[lab]})
train_rows.append({"text": "离婚后孩子抚养权如何判决", "label_id": 15}  )
train_rows.append({"text": "工伤认定需要什么条件", "label_id": 15}  )
train_rows.append({"text": "父母可以私自出售未成年子女的房产吗", "label_id": 15}  )
train_rows.append({"text": "公司裁员是否需要提前通知员工", "label_id": 15}  )
train_rows.append({"text": "交通事故责任划分后赔偿标准是什么", "label_id": 15}  )
train_rows.append({"text": "租房合同到期后能否自动续租", "label_id": 15}  )
train_rows.append({"text": "夫妻一方擅自贷款用于家庭开支是否有效", "label_id": 15}  )
train_rows.append({"text": "遗嘱未公证是否有效", "label_id": 15}  )
train_rows.append({"text": "网络诈骗案报案后多久能立案", "label_id": 15}  )
train_rows.append({"text": "劳动合同中约定试用期超过法定期限是否违法", "label_id": 15}  )
train_rows.append({"text": "usr: 我想了解一下离婚后财产分割的原则是怎样的？\nsys: 在离婚过程中，夫妻共同财产应当依法平均分割，但若存在婚前协议、财产约定或一方明显过错等情况，法院可酌情调整分割比例。", "label_id": 15}  )
train_rows.append({"text": "usr: 我被公司无故辞退，想知道我能申请哪些赔偿？\nsys: 根据《劳动合同法》，如果用人单位未提前30天通知解除劳动合同，需支付一个月工资作为代通知金，并可能需要支付经济补偿金，具体金额依据工作年限计算。", "label_id": 15})
train_rows.append({"text": "劳动合同中试用期最长不得超过多久", "label_id": 14}  )
train_rows.append({"text": "遗嘱公证的有效条件是什么", "label_id": 14}  )
train_rows.append({"text": "未成年人网络充值如何追回", "label_id": 14}  )
train_rows.append({"text": "婚姻关系存续期间取得的房产归属如何认定", "label_id": 14}  )
train_rows.append({"text": "工伤认定申请应在事故发生后多长时间内提出", "label_id": 14}  )
train_rows.append({"text": "公司股东抽逃出资的行为如何处罚", "label_id": 14}  )
train_rows.append({"text": "交通事故责任划分依据有哪些", "label_id": 14}  )
train_rows.append({"text": "农村土地承包经营权可以转让吗", "label_id": 14}  )
train_rows.append({"text": "消费者权益保护法中的‘三包’政策具体内容是什么", "label_id": 14}  )
train_rows.append({"text": "离婚冷静期制度实施后是否影响财产分割", "label_id": 14}  )
train_rows.append({"text": "usr: 请问我国《民法典》中关于相邻关系的规定有哪些？\nsys: 根据《民法典》第286条至第292条，相邻关系涉及不动产所有权人之间的通行、排水、采光、通风等方面的权利义务，例如相邻方不得堵塞通道、不得排放污染物影响他人正常生活等，具体需结合实际情况判断。", "label_id": 14}  )
train_rows.append({"text": "usr: 房屋租赁合同中押金退还的法律规定是什么？\nsys: 根据《民法典》第703条及第713条，承租人未按约定使用房屋或造成损失的，出租人有权扣除相应费用；若无违约行为，出租人应在租赁期满或合同解除后及时退还押金，且不得无故拖延或克扣。", "label_id": 14})
train_rows.append({"text": "杨老黑卤煮·京城老味的营业时间是什么？", "label_id": 13}  )
train_rows.append({"text": "北京全聚德烤鸭店的营业时间是多久？", "label_id": 13}  )
train_rows.append({"text": "护国寺小吃店每天几点开门？", "label_id": 13}  )
train_rows.append({"text": "三元梅园的营业时间是周一到周日吗？", "label_id": 13}  )
train_rows.append({"text": "老北京涮肉的营业时间表是多少？", "label_id": 13}  )
train_rows.append({"text": "厉家菜德胜门店几点关门？", "label_id": 13}  )
train_rows.append({"text": "姚记炒肝店鼓楼店的营业时间？", "label_id": 13}  )
train_rows.append({"text": "铃木食堂小菊儿胡同店什么时候营业？", "label_id": 13}  )
train_rows.append({"text": "聚宝源牛街总店的营业时间是怎样的？", "label_id": 13}  )
train_rows.append({"text": "北京海洋馆的开放时间是几点？", "label_id": 13}  )
train_rows.append({"text": "usr: 能告诉我北京全聚德烤鸭店的营业时间吗？\nsys: 好的，营业时间是周一至周日 11:30-14:00 17:00-21:00。", "label_id": 13}  )
train_rows.append({"text": "usr: 我想去看红砖美术馆，能告诉我它的开放时间吗？\nsys: 红砖美术馆的开放时间是每周二至周日 10:00-18:00（17:00停止入场）。", "label_id": 13})
train_rows.append({"text": "李白的性格特点是什么", "label_id": 12}  )
train_rows.append({"text": "林黛玉在红楼梦中的主要事迹", "label_id": 12}  )
train_rows.append({"text": "哪吒的性格设定和成长经历", "label_id": 12}  )
train_rows.append({"text": "花千骨的特殊能力来源", "label_id": 12}  )
train_rows.append({"text": "叶凡在斗破苍穹中的背景故事", "label_id": 12}  )
train_rows.append({"text": "白芷在古剑奇谭里的性格变化", "label_id": 12}  )
train_rows.append({"text": "苏轼在水浒传中如何影响剧情", "label_id": 12}  )
train_rows.append({"text": "小乔在三国演义中的婚恋关系", "label_id": 12}  )
train_rows.append({"text": "萧炎在斗罗大陆中的家庭背景", "label_id": 12}  )
train_rows.append({"text": "赵云在三国演义中的忠勇表现", "label_id": 12}  )
train_rows.append({"text": "usr: 能告诉我《诡秘之主》里克莱恩·莫雷蒂的性格吗？\nsys: 克莱恩·莫雷蒂是一个冷静、坚韧且富有同理心的角色，他在面对命运与力量时始终保持着内心的信念，是整个故事中极具深度的主角之一。", "label_id": 12}  )
train_rows.append({"text": "usr: 《庆余年》里的范闲是个什么样的人？\nsys: 范闲外表看似纨绔，实则聪慧过人、心思缜密，拥有极强的洞察力和反差魅力，是集智慧与胆识于一身的复杂人物。", "label_id": 12})
train_rows.append({"text": "主角在第几章觉醒了神力", "label_id": 11}  )
train_rows.append({"text": "女主最后和谁在一起了", "label_id": 11}  )
train_rows.append({"text": "小说里反派是怎么被打败的", "label_id": 11}  )
train_rows.append({"text": "主角穿越后获得了什么能力", "label_id": 11}  )
train_rows.append({"text": "书中的世界设定是什么样的", "label_id": 11}  )
train_rows.append({"text": "结局是开放式还是有明确收尾", "label_id": 11}  )
train_rows.append({"text": "主角和配角之间的感情线发展如何", "label_id": 11}  )
train_rows.append({"text": "第100章发生了什么重大转折", "label_id": 11}  )
train_rows.append({"text": "小说结尾有没有伏笔回收", "label_id": 11}  )
train_rows.append({"text": "主角在最后是否完成了自己的使命", "label_id": 11}  )
train_rows.append({"text": "usr: 能告诉我《诡秘之主》里克莱恩最后怎么样了吗？\nsys: 在故事的结局中，克莱恩最终选择了守护世界秩序，成为了一名真正的‘神’，并完成了自我救赎。", "label_id": 11}  )
train_rows.append({"text": "usr: 请问《庆余年》里范闲和林婉儿最后怎么了？\nsys: 范闲最终与林婉儿相守一生，两人在江湖纷争中找到了彼此的归属，过上了平静而幸福的生活。", "label_id": 11})
train_rows.append({"text": "微信小程序无法登录怎么办", "label_id": 10}  )
train_rows.append({"text": "photoshop打开文件报错如何解决", "label_id": 10}  )
train_rows.append({"text": "apple id登录不到苹果音乐怎么办", "label_id": 10}  )
train_rows.append({"text": "wps文档无法保存到云端是什么原因", "label_id": 10}  )
train_rows.append({"text": "chrome浏览器总是弹出广告怎么关闭", "label_id": 10}  )
train_rows.append({"text": "teams会议连接失败如何修复", "label_id": 10}  )
train_rows.append({"text": "钉钉无法发送消息提示网络错误", "label_id": 10}  )
train_rows.append({"text": "notion同步失败怎么手动恢复", "label_id": 10}  )
train_rows.append({"text": "手机百度地图搜索不出结果怎么办", "label_id": 10}  )
train_rows.append({"text": "vscode启动时报错无法加载插件", "label_id": 10}  )
train_rows.append({"text": "usr: 我的手机百度地图搜索不出结果，是怎么回事？\nsys: 建议您检查网络连接是否正常，也可以尝试清除缓存或重启应用，如果问题依旧，可能是地图数据更新延迟导致的，建议稍后再试。", "label_id": 10}  )
train_rows.append({"text": "usr: wps文档一直无法保存到云端，是不是账号问题？\nsys: 不是账号问题，可能是网络不稳定或云端存储空间不足，建议您检查网络状态，并确保已登录正确的WPS账号，同时尝试切换为离线模式后重新保存。", "label_id": 10})
train_rows.append({"text": "javascript async函数怎么用", "label_id": 9}  )
train_rows.append({"text": "react hooks的useEffect什么时候执行", "label_id": 9}  )
train_rows.append({"text": "python中如何实现多线程", "label_id": 9}  )
train_rows.append({"text": "spring boot如何配置数据库连接池", "label_id": 9}  )
train_rows.append({"text": "css中的flex布局原理是什么", "label_id": 9}  )
train_rows.append({"text": "node.js中express框架如何处理路由", "label_id": 9}  )
train_rows.append({"text": "java中static变量和实例变量的区别", "label_id": 9}  )
train_rows.append({"text": "vue组件通信有哪些方式", "label_id": 9}  )
train_rows.append({"text": "mongodb索引是如何提高查询效率的", "label_id": 9}  )
train_rows.append({"text": "go语言的goroutine和channel怎么使用", "label_id": 9}  )
train_rows.append({"text": "usr: 我在写一个后端接口，发现请求响应特别慢，是怎么回事？\nsys: 建议检查一下是否用了过多的数据库查询，或者有没有加缓存机制，比如Redis，可以显著提升性能。", "label_id": 9}  )
train_rows.append({"text": "usr: 我用Python写了个爬虫，运行起来总是崩溃，报错说内存不足，怎么办？\nsys: 可以尝试使用生成器（generator）来分批读取数据，避免一次性加载所有内容到内存中，这样能有效缓解内存压力。", "label_id": 9})
train_rows.append({"text": "宫保鸡丁的家常做法", "label_id": 8}  )
train_rows.append({"text": "如何在家做出正宗的川味麻婆豆腐", "label_id": 8}  )
train_rows.append({"text": "西红柿鸡蛋汤的正确做法和调味技巧", "label_id": 8}  )
train_rows.append({"text": "低脂版红烧肉怎么做才不油腻", "label_id": 8}  )
train_rows.append({"text": "素炒三丝的食材搭配和火候控制", "label_id": 8}  )
train_rows.append({"text": "自制泡菜的步骤和发酵条件", "label_id": 8}  )
train_rows.append({"text": "如何用电饭煲做完美味的白粥", "label_id": 8}  )
train_rows.append({"text": "清蒸鲈鱼的去腥和调味方法", "label_id": 8}  )
train_rows.append({"text": "糖醋排骨的酸甜比例和翻炒技巧", "label_id": 8}  )
train_rows.append({"text": "冬瓜排骨汤的炖煮时间和配料搭配", "label_id": 8}  )
train_rows.append({"text": "usr: 我想学做一道家常菜，有什么简单又美味的推荐吗？\nsys: 推荐你试试宫保鸡丁，做法简单，味道开胃，特别适合家庭聚餐哦！", "label_id": 8}  )
train_rows.append({"text": "usr: 家里没有酱油，能用其他调料代替宫保鸡丁里的酱汁吗？\nsys: 可以尝试用蚝油+醋+白糖+少许料酒混合代替酱油，味道也能保持类似风味！", "label_id": 8})
train_rows.append({"text": "蜜桃肌怎么打造", "label_id": 7}  )
train_rows.append({"text": "防晒霜到底要不要涂在眼周", "label_id": 7}  )
train_rows.append({"text": "如何在家做面膜改善暗沉", "label_id": 7}  )
train_rows.append({"text": "粉底液和气垫哪个更适合夏天", "label_id": 7}  )
train_rows.append({"text": "唇纹严重怎么办，有没有有效护理方法", "label_id": 7}  )
train_rows.append({"text": "用精华液代替面霜会不会伤皮肤", "label_id": 7}  )
train_rows.append({"text": "每天敷面膜会不会导致皮肤屏障受损", "label_id": 7}  )
train_rows.append({"text": "熬夜后皮肤泛红怎么快速修复", "label_id": 7}  )
train_rows.append({"text": "去角质频率过高对皮肤有危害吗", "label_id": 7}  )
train_rows.append({"text": "遮瑕膏和遮瑕笔哪个更持久", "label_id": 7}  )
train_rows.append({"text": "usr: 我最近皮肤总是泛红，晚上睡不好，有什么办法可以缓解吗？\nsys: 建议您使用含有积雪草成分的舒缓精华，每天早晚各一次，配合冷敷可以帮助缓解泛红现象。", "label_id": 7}  )
train_rows.append({"text": "usr: 想要打造自然裸妆效果，用什么产品组合比较好？\nsys: 推荐使用轻薄的气垫BB霜+无瑕遮瑕膏+素颜霜，搭配自然眉形和轻扫腮红即可完成裸妆哦！", "label_id": 7})
train_rows.append({"text": "高血压患者可以喝咖啡吗", "label_id": 6}  )
train_rows.append({"text": "感冒后咳嗽持续两周怎么办", "label_id": 6}  )
train_rows.append({"text": "糖尿病饮食注意事项有哪些", "label_id": 6}  )
train_rows.append({"text": "胃酸过多吃什么食物缓解", "label_id": 6}  )
train_rows.append({"text": "熬夜对心脏有什么影响", "label_id": 6}  )
train_rows.append({"text": "甲亢患者是否需要忌碘", "label_id": 6}  )
train_rows.append({"text": "腰痛可能是肾结石吗", "label_id": 6}  )
train_rows.append({"text": "儿童发烧38.5度该怎么办", "label_id": 6}  )
train_rows.append({"text": "失眠怎么调理才有效", "label_id": 6}  )
train_rows.append({"text": "便秘的原因及改善方法", "label_id": 6}  )
train_rows.append({"text": "usr: 我最近总是头晕，感觉心跳很快，去医院检查应该挂什么科？\nsys: 建议您挂心血管内科或者神经内科，根据具体症状进一步诊断。", "label_id": 6}  )
train_rows.append({"text": "usr: 我妈妈最近经常腹泻，还伴有腹痛，这种情况严重吗？\nsys: 这种情况可能与肠胃炎、消化不良或肠道感染有关，建议尽快就医，由医生判断是否需要做进一步检查。", "label_id": 6})
train_rows.append({"text": "晨跑对心脏有什么好处", "label_id": 5}  )
train_rows.append({"text": "如何通过饮食改善睡眠质量", "label_id": 5}  )
train_rows.append({"text": "老年人每天应该喝多少水", "label_id": 5}  )
train_rows.append({"text": "运动后肌肉酸痛是怎么回事", "label_id": 5}  )
train_rows.append({"text": "枸杞和红枣一起泡水有什么作用", "label_id": 5}  )
train_rows.append({"text": "便秘的常见原因及缓解方法", "label_id": 5}  )
train_rows.append({"text": "每天喝一杯蜂蜜水是否对身体有益", "label_id": 5}  )
train_rows.append({"text": "高血压患者可以吃咸菜吗", "label_id": 5}  )
train_rows.append({"text": "熬夜对免疫力的影响有多大", "label_id": 5}  )
train_rows.append({"text": "如何判断自己是否有颈椎病", "label_id": 5}  )
train_rows.append({"text": "usr: 我最近总是睡不好，有没有什么方法可以改善睡眠？\nsys: 建议您可以尝试睡前泡脚、避免看手机，保持规律作息，也可以适量饮用温牛奶或草本茶，有助于提升睡眠质量。", "label_id": 5}  )
train_rows.append({"text": "usr: 早上起来头晕，是不是血压低？该怎么调理？\nsys: 头晕可能与血压偏低有关，也可能是脱水或睡眠不足引起，建议多喝水、避免空腹运动，并适当补充盐分，若持续不适，建议就医检查。", "label_id": 5})
train_rows.append({"text": "usr: 请问北京故宫的详细地址是什么？", "label_id": 4}  )
train_rows.append({"text": "usr: 我想在朝阳区找一家地铁站附近的餐馆，能告诉我具体地址吗？", "label_id": 4}  )
train_rows.append({"text": "usr: 请问北京大学的主校区在哪个地址？", "label_id": 4}  )
train_rows.append({"text": "usr: 能帮我查一下天安门广场的准确地理位置吗？", "label_id": 4}  )
train_rows.append({"text": "usr: 有没有人知道北京奥林匹克森林公园的具体地址？", "label_id": 4}  )
train_rows.append({"text": "usr: 我要找一家在海淀区中关村附近、提供咖啡服务的店，地址是哪里？", "label_id": 4}  )
train_rows.append({"text": "usr: 请问北京协和医院的详细地址是？", "label_id": 4}  )
train_rows.append({"text": "usr: 想知道国家图书馆的地址，能不能告诉我？", "label_id": 4}  )
train_rows.append({"text": "usr: 有没有人知道中关村南大街1号的具体位置？", "label_id": 4}  )
train_rows.append({"text": "usr: 北京西站的地址是哪里？", "label_id": 4}  )
train_rows.append({"text": "usr: 请问上海外滩的地址是？\nsys: 上海外滩的地址是黄浦区中山东一路15号，靠近南京东路，是著名的旅游景点之一。", "label_id": 4}  )
train_rows.append({"text": "usr: 能帮我查一下杭州西湖的详细地址吗？\nsys: 杭州西湖的地址是浙江省杭州市西湖区龙井路，景区范围广泛，包含断桥、苏堤等多个著名景点。", "label_id": 4})
train_rows.append({"text": "北京朝阳区有哪些四星级以上的高端酒店？", "label_id": 3}  )
train_rows.append({"text": "上海浦东新区评分4.8分以上的五星级酒店有哪些？", "label_id": 3}  )
train_rows.append({"text": "广州天河区提供免费停车和健身房的高端酒店推荐？", "label_id": 3}  )
train_rows.append({"text": "杭州西湖边附近有没有性价比高且评分超过4.7的高端酒店？", "label_id": 3}  )
train_rows.append({"text": "成都宽窄巷子区域有哪些环境优雅、服务优质的高端酒店？", "label_id": 3}  )
train_rows.append({"text": "西安回民街周边有没有提供无烟房和家庭房的高端酒店？", "label_id": 3}  )
train_rows.append({"text": "南京夫子庙附近评分4.6分以上的高端酒店有哪些？", "label_id": 3}  )
train_rows.append({"text": "三亚亚龙湾有哪些临海视野好、配套齐全的高端酒店？", "label_id": 3}  )
train_rows.append({"text": "青岛五四广场附近有没有适合情侣入住的高端酒店？", "label_id": 3}  )
train_rows.append({"text": "重庆解放碑周边有哪些提供商务会议服务的高端酒店？", "label_id": 3}  )
train_rows.append({"text": "usr: 我想在成都宽窄巷子附近找一家高端酒店，能推荐一下吗？\nsys: 好的，推荐您入住成都宽窄巷子·锦江国际酒店，这家酒店评分4.9分，环境优美，交通便利，非常适合游客和商务人士。", "label_id": 3}  )
train_rows.append({"text": "usr: 请问北京国贸附近有哪些评分4.7分以上的高端酒店？\nsys: 推荐您选择北京国贸大酒店和北京丽思卡尔顿酒店，这两家都是四星级以上高端酒店，服务优质，位置优越，非常值得入住。", "label_id": 3})
train_rows.append({"text": "酒店周边有哪些餐馆呢", "label_id": 2}  )
train_rows.append({"text": "周边的餐馆有哪些推荐", "label_id": 2}  )
train_rows.append({"text": "去市中心附近能吃到地道小吃的餐馆有哪些", "label_id": 2}  )
train_rows.append({"text": "离地铁站最近的餐馆是哪家", "label_id": 2}  )
train_rows.append({"text": "有没有评分4.5以上的本地人推荐餐馆", "label_id": 2}  )
train_rows.append({"text": "附近有没有提供外卖服务的餐馆", "label_id": 2}  )
train_rows.append({"text": "想吃火锅，附近有哪些靠谱的店", "label_id": 2}  )
train_rows.append({"text": "有没有便宜又好吃的夜市摊位推荐", "label_id": 2}  )
train_rows.append({"text": "附近有没有中餐和日料都有的餐馆", "label_id": 2}  )
train_rows.append({"text": "请问这个小区附近有哪些连锁快餐店", "label_id": 2}  )
train_rows.append({"text": "usr: 我想去市中心逛一逛，你能推荐几家评分高的餐馆吗？\nsys: 当然可以，推荐您去姚记炒肝店（鼓楼店）、王胖子驴肉火烧（鼓楼店）、厉家菜(德胜门店)和老北京涮肉，都是本地人很爱的餐馆呢！", "label_id": 2}  )
train_rows.append({"text": "usr: 好的，酒店周边有哪些餐馆呢?\nsys: 周边餐馆有姚记炒肝店（鼓楼店）, 王胖子驴肉火烧（鼓楼店）, 厉家菜(德胜门店), 老北京涮肉, 三元梅园(鼓楼西大街店) ...，都是不错的餐馆呢！", "label_id": 2})
train_rows.append({"text": "王者荣耀打野路线怎么规划", "label_id": 0}  )
train_rows.append({"text": "原神水下探索技巧有哪些", "label_id": 0}  )
train_rows.append({"text": "绝地求生如何避免被狙击", "label_id": 0}  )
train_rows.append({"text": "CS2新手如何快速上手", "label_id": 0}  )
train_rows.append({"text": "和平精英地图视野优化技巧", "label_id": 0}  )
train_rows.append({"text": "英雄联盟打野gank时机判断", "label_id": 0}  )
train_rows.append({"text": "地铁跑酷滑行加速技巧", "label_id": 0}  )
train_rows.append({"text": "洛天依语音触发条件是什么", "label_id": 0}  )
train_rows.append({"text": "崩坏：星穹铁道角色站位策略", "label_id": 0}  )
train_rows.append({"text": "斗罗大陆手游魂师搭配技巧", "label_id": 0}  )
train_rows.append({"text": "usr: 我在玩英雄联盟，打野的时候怎么判断最佳gank时机？\nsys: 打野gank的最佳时机通常是在敌方野区刷新后10秒内，尤其是对方补兵或回城时，配合视野控制可以提高成功率。", "label_id": 0}  )
train_rows.append({"text": "usr: 原神深渊挑战怎么才能打得更稳？\nsys: 深渊挑战建议提前规划队伍搭配，合理利用元素反应，保持站位安全，同时注意资源管理，避免被敌人集火。", "label_id": 0})
train_rows.append({"text": "dnf525改版职业排行", "label_id": 1}  )
train_rows.append({"text": "原神水元素角色哪个最强", "label_id": 1}  )
train_rows.append({"text": "暗黑破坏神不朽射手职业出装推荐", "label_id": 1}  )
train_rows.append({"text": "英雄联盟亚索怎么玩才能打野效率最高", "label_id": 1}  )
train_rows.append({"text": "崩坏星穹铁道五星角色搭配推荐", "label_id": 1}  )
train_rows.append({"text": "王者荣耀张飞的皮肤有哪些以及获取方式", "label_id": 1}  )
train_rows.append({"text": "最终幻想14白魔法师职业性格设定", "label_id": 1}  )
train_rows.append({"text": "艾尔登法环雷电之神的背景故事和技能", "label_id": 1}  )
train_rows.append({"text": "明日方舟角色图鉴中v.v.的特殊事迹", "label_id": 1}  )
train_rows.append({"text": "塞尔达传说荒野之息林克的战斗风格分析", "label_id": 1}  )
train_rows.append({"text": "usr: 我想了解下原神里水元素角色的强度排名，能给我推荐几个吗？\nsys: 当前版本中，水元素角色中‘菲谢尔’和‘诺艾尔’表现非常稳定，尤其在团队辅助和控制方面优势明显，推荐优先考虑哦！", "label_id": 1}  )
train_rows.append({"text": "usr: 请问王者荣耀里张飞的皮肤有哪些，怎么获得？\nsys: 张飞目前有‘武圣出征’‘战神降临’‘狂风怒吼’三个皮肤，其中‘战神降临’可以通过活动限时获取，其他可通过抽取获得。", "label_id": 1})
train_rows.append({"text": "火麻仁的功效与作用及食用方法", "label_id": 5})
train_rows.append({"text": "usr: 你好，我想找一个酒店住宿，可以帮我推荐一个吗？\nsys: 给您推荐北京贵都大酒店，速8酒店(北京前门大栅栏店)和北京西单饭店，这些都是很不错的酒店呢！\nusr: 好的，就去北京贵都大酒店吧，能帮我查一下这里的电话吗？\nsys: 没问题，电话是010-51979888。\nusr: 收到，请问这个酒店周边都有哪些餐馆？\nsys: 周边餐馆有北京全聚德, 东来顺(前门大街店), 便宜坊烤鸭(鲜鱼口店), 聚宝源(牛街总店), 都一处烧麦馆(前门店) 等，都是老字号了。\nusr: 哇，这么多好吃的地方呀，酒店的酒店设施提供无烟房吗？\nsys: 是的，酒店的酒店设施提供无烟房。\nusr: 谢谢，我想要去一个游玩时长1小时 - 2小时，评分是4.5分以上的景点游玩，有什么好的推荐吗？\nsys: 红砖美术馆，将府公园和钟鼓楼都是很不错的选择呢！\nusr: 就去钟鼓楼吧，景点周边都有什么酒店呢？\nsys: 周边酒店有北京天伦王朝酒店, 北京都季商旅酒店, 北京香江戴斯酒店, 北京饭店。\nusr: 这多酒店呀，那景点周边有哪些餐馆呢？\nsys: 好多呢，周边餐馆有姚记炒肝店（鼓楼店）, 护国寺小吃店（护国寺总店）, 北京全聚德(王府井店), 铃木食堂(小菊儿胡同店)。\nusr: 我打算去钟鼓楼附近的酒店住宿，这些酒店哪家最低价格是200-300元的呢？\nsys: 对不起，没有查到符合您需求的酒店，推荐您去北京香江戴斯酒店住宿，这是最接近您需求的酒店了，在钟鼓楼附近，价格328元。\nusr: 可以，就去北京香江戴斯酒店吧，这个酒店评分是多少呢？\nsys: 酒店评分4.2分。\nusr: 好的，酒店周边有哪些餐馆呢？\nsys: 周边餐馆有姚记炒肝店（鼓楼店）, 王胖子驴肉火烧（鼓楼店）, 厉家菜(德胜门店), 老北京涮肉, 三元梅园(鼓楼西大街店) ...，都是不错的餐馆呢！", "label_id": 2})
train_rows.append({"text": "丙氨酸氨基转移酶偏高的原因", "label_id": 6})
train_rows.append({"text": "usr: 你好啊，你知道哪家餐馆的人均消费是50-100元，评分还是4.5分以上的吗？可以给我推荐一下吗？\nsys: 去吃披萨怎么样，我推荐一家站点比萨(北大餐厅)。\nusr: 行啊，我挺喜欢吃披萨的，他家的地址发给我吧。\nsys: 好的，地址是福缘门1号。\nusr: 行，吃过饭，接下来我得出去走走了，你看哪个景点是评分在4分以上的，门票价格是150-200元的啊，介绍给我呗。\nsys: 我提议你去北京海洋馆看看，看完你的心情一定好很多。\nusr: 好的，就听你的，这个景点的周边有餐馆没？把他家的电话发给我吧。\nsys: 他家的周边餐馆挺多呢，如海碗居北京菜(增光路总店), 护国寺小吃店（护国寺总店）, 姚记炒肝店（鼓楼店）, 铃木食堂(小菊儿胡同店)，你可以有很多选择，我把电话发给你啊，010-62176655。\nusr: 那就麻烦你在上述的周边餐馆里找一家评分是4.5分以上的，有刀削面可以吃的餐馆给我好吗？\nsys: 我没有找到符合你的要求的餐馆，不过我看京海大厦餐厅也很好，不如你换个口味去吃清蒸鲈鱼怎么样？\nusr: 你的建议很好啊，把他家的地址给我吧。\nsys: 好的，地址是翠微路38号。", "label_id": 4})
train_rows.append({"text": "陈长生和落落最后咋了", "label_id": 11})
train_rows.append({"text": "鬼见愁是谁", "label_id": 12})
train_rows.append({"text": "usr: 你好，你能帮我找一个游玩时长是0.3小时 - 0.5小时的景点吗？\nsys: 没问题，我建议你去祈年殿和北京古代建筑博物馆。\nusr: 好的，就去北京古代建筑博物馆，帮我查查这个景点周边的酒店都有什么可以吗？\nsys: 当然了，周边酒店有北京贵都大酒店, 北京首都宾馆, 北京饭店, 北京天伦王朝酒店。\nusr: 住的地方还不少呢，可以把景点的地址给我吗？\nsys: 那你记一下吧，地址是北京市西城区东经路21号。\nusr: 好的，我记下了，溜达完我需要找一个地方落脚，给我安排一个最低价格200-300元，提供行李寄存的舒适型酒店可以吗？\nsys: 我觉得山水时尚酒店(北京前门店)和北京梧桐树酒店都还挺好的，给你做参考吧。\nusr: 嗯，就去山水时尚酒店(北京前门店)吧，告诉我一下他家电话。\nsys: 可以，电话是010-63048088。\nusr: 收到，安排好住处我打算先去吃点东西，哪个餐馆评分是4.5分以上，推荐菜有干炸丸子啊？\nsys: 给你提供2个选择杏园餐厅和杨老黑卤煮·京城老味，反正我觉得还行，你自己拿主意吧。\nusr: 那我选择去吃杨老黑卤煮·京城老味，麻烦你告诉我一下他家的营业时间。\nsys: 好的，营业时间是周一至周日 10:30-22:00。", "label_id": 13})
train_rows.append({"text": "usr: 你好可以帮我找一个价格1000元以上，提供宽带上网的高档型酒店吗？\nsys: 推荐您去北京贵都大酒店住宿，这家完全符合您的要求。\nusr: 可以啊，你知道这个酒店在什么地方吗？\nsys: 这家酒店的地址是在北京西城区广安门内大街217号。\nusr: 好的，定好酒店以后我打算找个人均消费是100-150元，评分是4.5分以上的餐馆吃饭，有什么好的地方给我推荐吗？\nsys: 推荐你去全聚德烤鸭店(清华园店)用餐，这家餐馆做菜口碑很不错呢！\nusr: 行啊，就去这里吃吧，它家周边还有别的餐馆吗？\nsys: 周边的餐馆还有小吊梨汤(畅春园店), 白家大院。\nusr: 好的，它家一般几点营业啊？\nsys: 营业时间是周一 至 周日 11:30-14:00 17:00-21:00。\nusr: 好的，这个时间段还是蛮方便的，吃完饭我想先找个评分是5分，游玩时长是1小时的景点散步，有啥好的建议吗？\nsys: 推荐您去红砖美术馆这个景点游玩，完全符合您的要求呢！\nusr: 可以啊，正好我是学美术的，你知道这个景点的电话吗？\nsys: 这个景点的电话是010-84576669。\nusr: 嗯好，我知道了，那这个景点周边有餐馆吗？\nsys: 周边的餐馆有BHG Kitchen(欧陆广场店), 本家韩国料理(望京店)。\nusr: 真不错，我打算游玩结束以后在景点周边找个能做麻辣馋嘴蛙的餐馆大吃一顿，知道哪个比较适合我吗？\nsys: 抱歉，周边的餐馆都没有这道菜呢，推荐您去非周边的餐馆四川人家用餐这道菜。\nusr: 那好吧，只要能吃到这个菜就行，知道这里的人均消费是多少吗？\nsys: 这家餐馆的人均消费是102元。\nusr: OK，这个几位还可以，麻烦你再帮我查一下这个餐馆周边都有什么酒店好吗？\nsys: 周边的酒店有北京丰大国际大酒店, 北京亦庄智选假日酒店。", "label_id": 3})
train_rows.append({"text": "农村房屋继承法新规定", "label_id": 14})
train_rows.append({"text": "商住楼50年后怎么办", "label_id": 15})
train_rows.append({"text": "lol定位赛10场全胜什么段位", "label_id": 0})
train_rows.append({"text": "dnf525改版职业排行", "label_id": 1})
train_rows.append({"text": "吸油纸对皮肤有害吗", "label_id": 7})
train_rows.append({"text": "越南春卷的做法", "label_id": 8})
train_rows.append({"text": "error launching installer英雄联盟", "label_id": 10})
train_rows.append({"text": "overflow hidden什么意思", "label_id": 9})
print(f"训练样本: {len(train_rows)}")
for row in train_rows:
    print(f"label_id: {row['label_id']} (type: {type(row['label_id'])})")

In [55]:
from transformers import BertTokenizer, BertModel
class IntentDataset(Dataset):
    def __init__(self, rows, tokenizer, max_len, with_label=True):
        self.rows = rows; self.tokenizer = tokenizer; self.max_len = max_len; self.with_label = with_label

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows[idx]
        enc = self.tokenizer(row["text"], add_special_tokens=True, truncation=True,
                             max_length=self.max_len, padding="max_length", return_tensors="pt")
        item = {"input_ids": enc["input_ids"].flatten(), "attention_mask": enc["attention_mask"].flatten()}
        if self.with_label:
            item["labels"] = torch.tensor(row["label_id"], dtype=torch.long)
        return item


class BertIntentClassifier(nn.Module):
    def __init__(self, model_name, num_classes):
        super().__init__()
        self.bert = BertModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        if isinstance(out, tuple):
            pooled = out[0][:, 0]
        else:
            pooled = getattr(out, "pooler_output", None)
            if pooled is None:
                pooled = out.last_hidden_state[:, 0]
        return self.classifier(self.dropout(pooled))


tokenizer = BertTokenizer.from_pretrained(MODEL_DIR)
model = BertIntentClassifier(MODEL_DIR, len(INTENT_LABELS)).to(DEVICE)

train_loader = DataLoader(IntentDataset(train_rows, tokenizer, MAX_LEN), batch_size=BATCH_SIZE, shuffle=True)
optim = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
total_steps = max(1, len(train_loader) * EPOCHS)
scheduler_cosine = CosineAnnealingLR(optim, T_max=10, eta_min=0)

warmup_epochs = 5
warmup_lr = 0.00002
initial_lr = 0.000001
# 自定义学习率预热函数
def warmup_lambda(current_step):
    if current_step < warmup_epochs * len(train_loader):
        return float(current_step) / (warmup_epochs * len(train_loader))
    else:
        return 1.0
        
scheduler_warmup = LambdaLR(optim, lr_lambda=warmup_lambda)

current_step = 0
for ep in range(EPOCHS):
    print(f"epoch {ep+1}/{EPOCHS}")
    model.train()
    for batch in tqdm(train_loader, desc="train", leave=False):
        optim.zero_grad()
        x = batch["input_ids"].to(DEVICE); m = batch["attention_mask"].to(DEVICE); y = batch["labels"].to(DEVICE)
        loss = nn.CrossEntropyLoss()(model(x, m), y)
        loss.backward(); optim.step();
        if ep < warmup_epochs:
            scheduler_warmup.step()
        else:
            scheduler_cosine.step()

        current_step += 1
print("training done.")

In [ ]:
model.eval()
import os

if os.environ.get('DATA_PATH'):
    DATA_PATH = os.environ.get("DATA_PATH") + "/"  
else:
    print("Baseline运行时，因为无法读取测试集，所以会有此条报错，属于正常现象")  
    print("When baseline is running, this error message will appear because the test set cannot be read, which is a normal phenomenon.")

TEST_A_PATH = Path(DATA_PATH + "/val.jsonl")
TEST_B_PATH = Path(DATA_PATH + "/test.jsonl")


@torch.no_grad()
def predict_to_file(test_path, out_path):
    model.eval()
    records = load_jsonl(test_path)
    rows = [{"text": preprocess_input_text(r.get("input")) or ""} for r in records]
    loader = DataLoader(IntentDataset(rows, tokenizer, MAX_LEN, with_label=False), batch_size=BATCH_SIZE*4, shuffle=False)
    preds = []
    for batch in tqdm(loader, desc="predict", leave=False):
        x = batch["input_ids"].to(DEVICE); m = batch["attention_mask"].to(DEVICE)
        preds.extend(torch.argmax(model(x, m), dim=1).cpu().tolist())
    assert len(preds) == len(records)
    with Path(out_path).open("w", encoding="utf-8") as fh:
        for p in preds:
            fh.write(json.dumps({"label": ID_TO_LABEL[p]}, ensure_ascii=False) + "\n")
    print(f"wrote {len(preds)} -> {out_path}")


predict_to_file(TEST_A_PATH, Path("submission_val.jsonl"))
predict_to_file(TEST_B_PATH, Path("submission_test.jsonl"))

In [ ]:
SUBMISSION_ZIP = Path("submission.zip")
with zipfile.ZipFile(SUBMISSION_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write("submission_val.jsonl")
    zf.write("submission_test.jsonl")
print(f"wrote {SUBMISSION_ZIP} ({SUBMISSION_ZIP.stat().st_size} bytes)")

In [ ]:
!ls -l /bohr/train-z7v5/v1/
!ls -l
!cp /bohr/train-z7v5/v1/train.jsonl /personal/
!cp model_parameters.pth /personal/